# Day 1 — RAG Fundamentals: Retrieval Pipeline + Local LLM with Ollama

**Retrieve → Augment → Generate**, built step by step:

| Part | What we build | Runs on |
|---|---|---|
| 1 | Knowledge base, embeddings, similarity search, `retrieve()` | `sentence-transformers` + NumPy (local) |
| 2 | Prompt template (the *augment* step) | plain Python |
| 3 | Answer generation with a local LLM | **Ollama** (local, no API key) |

```
Documents → Embeddings → Query encoding → Cosine similarity → Ranking → Top-K
          → Context assembly → Prompt → Ollama LLM → Grounded answer + sources
```

> **Teaching tip:** run the notebook top to bottom, one cell at a time. Markdown cells marked 🎓 are talking points.

## 0 · Setup (do this before class)

**1. Python environment** (Windows PowerShell; on macOS/Linux use `source .venv/bin/activate`):

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install sentence-transformers numpy "ollama>=0.4" jupyter
python -m ipykernel install --user --name rag-day1   # makes the venv selectable as a kernel
jupyter notebook
```

**2. Ollama** — install from <https://ollama.com/download>, then in a terminal:

```powershell
ollama pull llama3.2      # ~2 GB, small model that runs on a laptop CPU
ollama list               # confirm the model is there
```

Ollama runs a local server at `http://localhost:11434`. The desktop app starts it automatically; otherwise run `ollama serve`.

**3. First run needs internet** to download the embedding model `all-MiniLM-L6-v2` (~90 MB) and the Ollama model. Afterwards everything works offline.

| Problem | Fix |
|---|---|
| `Activate.ps1 cannot be loaded` | `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`, reopen terminal |
| Notebook can't import `sentence_transformers` | Select the **rag-day1** kernel (Kernel → Change kernel) |
| `Connection refused` to Ollama | Start the Ollama app or run `ollama serve` |
| `model "llama3.2" not found` | `ollama pull llama3.2`, or set `OLLAMA_MODEL` to a model from `ollama list` |

## 1 · Imports and configuration

In [ ]:
import os

import numpy as np
from sentence_transformers import SentenceTransformer

# Embedding model: used for BOTH documents and queries
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Ollama settings - change with environment variables, no code edits needed
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")   # any model shown by `ollama list`

print("Embedding model:", EMBED_MODEL_NAME)
print("Ollama:", OLLAMA_HOST, "| model:", OLLAMA_MODEL)

## 2 · Define the knowledge base

Real systems create chunks with an ingestion pipeline (extract → clean → chunk). Today we write six small, already-chunked policy passages by hand.

Each chunk has:
- **`id`** – stable identifier (trace, update, delete later)
- **`text`** – what gets embedded and sent to the LLM
- **`source`** – needed for citations
- **`metadata`** – extra fields for filtering later

In [ ]:
KNOWLEDGE_BASE = [
    {"id": "HR-001", "text": "Employees receive 18 days of annual paid leave per calendar year.",
     "source": "hr_leave_policy.pdf", "metadata": {"department": "HR", "updated": "2026-01-15"}},
    {"id": "HR-002", "text": "Remote work is permitted up to three days per week with manager approval.",
     "source": "remote_work_policy.md", "metadata": {"department": "HR", "updated": "2026-03-02"}},
    {"id": "FIN-001", "text": "Expense claims must be submitted within 30 days of the transaction date.",
     "source": "finance_expenses.pdf", "metadata": {"department": "Finance", "updated": "2025-11-20"}},
    {"id": "IT-001", "text": "The IT helpdesk operates from 9 AM to 6 PM on weekdays.",
     "source": "it_support_guide.txt", "metadata": {"department": "IT", "updated": "2026-02-10"}},
    {"id": "IT-002", "text": "Employees must change their network password every 90 days.",
     "source": "it_security_policy.pdf", "metadata": {"department": "IT", "updated": "2026-04-01"}},
    {"id": "FAC-001", "text": "Visitors must sign in at reception and wear a visitor badge at all times.",
     "source": "office_facilities.md", "metadata": {"department": "Facilities", "updated": "2025-09-05"}},
]

for chunk in KNOWLEDGE_BASE:
    print(f"{chunk['id']:<8} {chunk['source']:<24} {chunk['text']}")

## 3 · Load the embedding model

Loading takes a few seconds — do it **once**, never inside a per-query function. The first run downloads the model.

In [ ]:
try:
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Could not load '{EMBED_MODEL_NAME}': {exc}\n"
        "The first run needs internet access to download the model."
    ) from exc

print("Embedding dimension:", embed_model.get_sentence_embedding_dimension())

## 4 · Embed every chunk (indexing)

`normalize_embeddings=True` scales each vector to length 1 → **dot product = cosine similarity**.

🎓 *Ask:* what shape do you expect? (6 chunks × 384 numbers)

In [ ]:
def build_index(model, chunks):
    if not chunks:
        raise ValueError("The knowledge base is empty - add at least one chunk.")
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return embeddings.astype(np.float32)          # shape: (num_chunks, 384)


doc_embeddings = build_index(embed_model, KNOWLEDGE_BASE)

print("Shape:", doc_embeddings.shape)
print("Row lengths (should all be 1.0):", np.linalg.norm(doc_embeddings, axis=1).round(3))
print("First 8 numbers of HR-001:", doc_embeddings[0][:8].round(3))

🎓 The numbers are **not readable text**. That's why we keep `text` and `source` next to the vectors — after search we need the words for the LLM and the citation.

## 5 · Similarity search by hand — one query, step by step

Before wrapping it in a function, look at every intermediate value.

In [ ]:
query = "How many annual leave days do employees receive?"

# 1) Encode the query with the SAME model and normalisation
query_vec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
print("query_vec shape:", query_vec.shape)

# 2) One matrix-vector product scores every chunk: (6, 384) @ (384,) -> (6,)
scores = doc_embeddings @ query_vec
print("scores:", scores.round(3))

# 3) Rank: argsort is ascending, [::-1] makes it best-first
order = np.argsort(scores)[::-1]
print("\nRanking:")
for rank, idx in enumerate(order, start=1):
    print(f"  #{rank} {KNOWLEDGE_BASE[idx]['id']:<8} {scores[idx]:.3f}  {KNOWLEDGE_BASE[idx]['text']}")

🎓 Scores are **similarities, not probabilities**. Compare them within one model and dataset; don't read 0.6 as "60 % sure".

## 6 · A reusable `retrieve()` function

In [ ]:
def retrieve(query, top_k=3, embeddings=None, chunks=None):
    # Return the top_k most similar chunks with score and source metadata.
    # Uses the current global index unless another one is passed in.
    embeddings = doc_embeddings if embeddings is None else embeddings
    chunks = KNOWLEDGE_BASE if chunks is None else chunks
    if not query or not query.strip():
        raise ValueError("Query must be a non-empty string.")
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    top_k = min(top_k, len(chunks))

    query_vec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = embeddings @ query_vec
    ranked = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "rank": rank,
            "id": chunks[i]["id"],
            "score": float(scores[i]),
            "text": chunks[i]["text"],
            "source": chunks[i]["source"],
            "metadata": chunks[i]["metadata"],
        }
        for rank, i in enumerate(ranked, start=1)
    ]


def print_results(query, results):
    print(f"\nQuery: {query}")
    for r in results:
        print(f"  #{r['rank']}  {r['id']:<8} score={r['score']:.3f}  source={r['source']}")
        print(f"       {r['text']}")

## 7 · Run example queries

In [ ]:
for q in [
    "How many annual leave days do employees receive?",
    "Can I work from home?",                     # shares almost no words with the remote-work policy
    "What is the deadline for submitting expense claims?",
    "When can I contact IT support?",
]:
    print_results(q, retrieve(q, top_k=3))

### 7.1 · A question the knowledge base **cannot** answer

In [ ]:
print_results("What is the company's maternity leave policy?", retrieve("What is the company's maternity leave policy?", top_k=3))

🎓 Retrieval **always** returns the K nearest chunks — it never says "nothing found". The annual-leave chunk is *related* (leave, employees) but does **not** answer the question.

There is **no universal score threshold**; it must be validated on labelled questions for your model and data. In Section 9 we add the second safeguard: an LLM instruction to abstain when the context lacks the answer.

## 8 · Augment: build the prompt

Plain string building. The instructions do three jobs:
1. use **only** the context,
2. **abstain** with a fixed sentence when the answer is missing,
3. **cite** sources.

In [ ]:
INSUFFICIENT = "Insufficient information in the available documents."

SYSTEM_PROMPT = (
    "You are a company policy assistant. "
    "Answer the question using ONLY the context provided. "
    f'If the context does not contain the answer, reply exactly: "{INSUFFICIENT}" '
    "Cite the source of every fact in square brackets, e.g. [hr_leave_policy.pdf]. "
    "Keep the answer short."
)


def build_prompt(query, results):
    context = "\n".join(f"[{r['source']}] {r['text']}" for r in results)
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"


example = retrieve("How many annual leave days do employees receive?", top_k=2)
print("SYSTEM:\n" + SYSTEM_PROMPT + "\n")
print("USER:\n" + build_prompt("How many annual leave days do employees receive?", example))

## 9 · Generate: connect to Ollama

The `ollama` Python package talks to the local Ollama server. Nothing leaves your machine and no API key is needed.

First check that the server is running and the model is pulled.

In [ ]:
import ollama

client = ollama.Client(host=OLLAMA_HOST)

try:
    available = [m.model for m in client.list().models]
    print("Ollama is running. Local models:", available)
    if not any(name.split(":")[0] == OLLAMA_MODEL.split(":")[0] for name in available):
        print(f"\n⚠️  '{OLLAMA_MODEL}' is not pulled yet. Run in a terminal:  ollama pull {OLLAMA_MODEL}")
except Exception as exc:
    print(f"❌ Cannot reach Ollama at {OLLAMA_HOST}: {exc}")
    print("Start the Ollama desktop app or run `ollama serve`, then re-run this cell.")

In [ ]:
def generate_answer(query, results, model=OLLAMA_MODEL, temperature=0.0):
    # Send the system instructions + context + question to the local LLM.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(query, results)},
    ]
    try:
        response = client.chat(
            model=model,
            messages=messages,
            options={"temperature": temperature},   # 0 = as deterministic as possible
        )
    except ollama.ResponseError as exc:          # e.g. model not found
        raise RuntimeError(f"Ollama error: {exc.error}. Try: ollama pull {model}") from exc
    except Exception as exc:                     # server not running, network error
        raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST}: {exc}") from exc
    return response.message.content.strip()

🎓 **Temperature 0** makes answers repeatable — good for policy Q&A and for demos. Still, LLM output can vary slightly between model versions.

## 10 · The complete RAG pipeline: `rag_answer()`

Retrieve → Augment → Generate, and return the sources the answer was based on.

In [ ]:
def rag_answer(query, top_k=3, show_context=True):
    results = retrieve(query, top_k=top_k)
    answer = generate_answer(query, results)

    print(f"❓ {query}")
    if show_context:
        print("\n📚 Retrieved context:")
        for r in results:
            print(f"   {r['id']:<8} {r['score']:.3f}  [{r['source']}] {r['text']}")
    print(f"\n🤖 {answer}")
    if INSUFFICIENT.lower().rstrip(".") not in answer.lower():
        print("\n📎 Sources considered:", ", ".join(sorted({r["source"] for r in results})))
    print("-" * 80)
    return {"question": query, "answer": answer, "sources": results}


_ = rag_answer("How many annual leave days do employees receive?")

### 10.1 · More questions — including the ones that should be refused

In [ ]:
for q in [
    "Can I work from home on Fridays?",
    "What is the company's maternity leave policy?",            # not in the knowledge base -> should abstain
    "What are the rules for remote work and expense claims?",   # needs TWO chunks
]:
    _ = rag_answer(q, top_k=3)

🎓 Discussion points:
- **Maternity question:** retrieval still returned the leave chunk, but the instruction lets the LLM abstain. Small local models *sometimes* ignore instructions — that is why RAG **reduces** hallucination risk but does not remove it. Production systems add score thresholds (validated), re-ranking, citation checks and evaluation.
- **Multi-part question:** try `top_k=1` below — one of the two policies is missing from the context, so the answer can only be partial.

In [ ]:
_ = rag_answer("What are the rules for remote work and expense claims?", top_k=1)

## 11 · Exercises

**Exercise 1 — Add knowledge without training.** Append the two chunks below, rebuild the index, and ask *"How long is the trial period for new starters?"* and *"Can I fly business class to New York?"*

**Exercise 2 — Change Top-K.** Run the multi-part question with `top_k=1, 2, 3` and compare the answers.

**Exercise 3 — Probe the limits.** Ask *"How many sick days do employees get?"* and *"What is the parking policy?"*. Which chunks are retrieved? Does the model abstain? Try another model, e.g. `generate_answer(q, results, model="mistral")` after `ollama pull mistral`.

In [ ]:
# Exercise 1
KNOWLEDGE_BASE += [
    {"id": "HR-003", "text": "New employees complete a probation period of six months.",
     "source": "hr_onboarding.pdf", "metadata": {"department": "HR", "updated": "2026-05-12"}},
    {"id": "FIN-002", "text": "Business-class flights are only allowed for journeys longer than eight hours.",
     "source": "finance_travel_policy.pdf", "metadata": {"department": "Finance", "updated": "2026-06-01"}},
]
doc_embeddings = build_index(embed_model, KNOWLEDGE_BASE)   # re-index! skip this line and the new chunks stay invisible
print("New index shape:", doc_embeddings.shape)

for q in ["How long is the trial period for new starters?", "Can I fly business class to New York?"]:
    _ = rag_answer(q, top_k=2)

In [ ]:
# Exercise 2
q = "What are the rules for remote work and expense claims?"
for k in (1, 2, 3):
    print(f"===== top_k = {k} =====")
    _ = rag_answer(q, top_k=k, show_context=False)

In [ ]:
# Exercise 3
for q in ["How many sick days do employees get?", "What is the parking policy?"]:
    _ = rag_answer(q, top_k=3)

## 12 · Recap

| Step | Code | Output |
|---|---|---|
| Index | `build_index()` | `(N, 384)` normalised vectors |
| Retrieve | `retrieve()` | Top-K chunks with scores + sources |
| Augment | `build_prompt()` + `SYSTEM_PROMPT` | instructions + context + question |
| Generate | `generate_answer()` via Ollama | answer with citations, or *insufficient information* |

**Key points**
- Use the **same embedding model** for documents and queries.
- Similarity ≠ correctness: retrieval always returns *something*.
- The LLM only sees what you put in the prompt — retrieval quality caps answer quality.
- RAG reduces hallucination risk; it does not eliminate it.

**Next:** better ingestion and chunking, evaluation, and serving `rag_answer()` behind a FastAPI endpoint in Docker.